# FULL DATASET ANALYSIS (POLARS)

This notebook contains comprehensive analysis for the full datasets using **Polars** for optimized performance

**Datasets:**
- train_full: 5,337,414 rows, 94 columns
- test_full: 1,447,107 rows, 92 columns

**Analysis Pipeline:**
1. Data identification and structure
2. Missing values analysis
3. Distribution analysis (all types)
4. Outlier detection
5. Data type optimization
6. Standardization strategies

**Polars Benefits:**
- Faster data loading and processing
- Memory efficient operations
- Lazy evaluation capabilities
- Better multi-threading performance

## 1.1 IMPORTS AND CONFIGURATION

Setting up the environment with all necessary libraries and Polars configuration for optimal performance.

In [45]:
# ============================================
# IMPORTS AND CONFIGURATION
# ============================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
import polars as pl
from pathlib import Path
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import cohen_kappa_score
import time
from collections import Counter

# Polars configuration
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)

# Set up paths
base_dir = Path("..")
full_train_path = base_dir / "data" / "train.parquet"
full_test_path = base_dir / "data" / "test.parquet"

print("✅ Imports and configuration complete")

✅ Imports and configuration complete


## 1.2 DATA LOADING

Loading the full train and test datasets using Polars with performance timing.

In [46]:
# ============================================
# LOAD FULL DATASETS
# ============================================
print("="*60)
print("LOADING FULL DATASETS")
print("="*60)

# Load full datasets using Polars
start_time = time.time()
train_full = pl.read_parquet(full_train_path)
test_full = pl.read_parquet(full_test_path)
load_time = time.time() - start_time

print(f"Train full: {train_full.shape}")
print(f"Test full: {test_full.shape}")
print(f"Load time: {load_time:.2f} seconds")

print(f"\n✅ Full datasets loaded")
print(f"   Train: {train_full.shape}")
print(f"   Test: {test_full.shape}")

LOADING FULL DATASETS
Train full: (5337414, 94)
Test full: (1447107, 92)
Load time: 13.62 seconds

✅ Full datasets loaded
   Train: (5337414, 94)
   Test: (1447107, 92)


## 2.0 DATA STRUCTURE ANALYSIS

Comprehensive analysis of dataset structure, column types, and basic statistics.

In [6]:
# ============================================
# BASIC DATA STRUCTURE ANALYSIS
# ============================================
print("="*60)
print("DATA STRUCTURE ANALYSIS")
print("="*60)

# Train dataset structure
print("\n TRAIN DATASET STRUCTURE:")
print(f"Shape: {train_full.shape}")
print(f"Columns: {train_full.columns}")
print(f"\nData types:")
# Correct way to get dtype counts in Polars
train_dtype_counts = Counter(str(dtype) for dtype in train_full.dtypes)
for dtype, count in train_dtype_counts.items():
    print(f"{dtype}: {count}")

# Test dataset structure  
print("\n TEST DATASET STRUCTURE:")
print(f"Shape: {test_full.shape}")
print(f"Columns: {test_full.columns}")
print(f"\nData types:")
# Correct way to get dtype counts in Polars
test_dtype_counts = Counter(str(dtype) for dtype in test_full.dtypes)
for dtype, count in test_dtype_counts.items():
    print(f"{dtype}: {count}")

# Column differences
train_cols = set(train_full.columns)
test_cols = set(test_full.columns)
only_in_train = train_cols - test_cols
only_in_test = test_cols - train_cols

print(f"\n COLUMN DIFFERENCES:")
print(f"Only in train: {only_in_train}")
print(f"Only in test: {only_in_test}")
print(f"Common columns: {len(train_cols & test_cols)}")

DATA STRUCTURE ANALYSIS

📋 TRAIN DATASET STRUCTURE:
Shape: (5337414, 94)
Columns: ['id', 'code', 'sub_code', 'sub_category', 'horizon', 'ts_index', 'feature_a', 'feature_b', 'feature_c', 'feature_d', 'feature_e', 'feature_f', 'feature_g', 'feature_h', 'feature_i', 'feature_j', 'feature_k', 'feature_l', 'feature_m', 'feature_n', 'feature_o', 'feature_p', 'feature_q', 'feature_r', 'feature_s', 'feature_t', 'feature_u', 'feature_v', 'feature_w', 'feature_x', 'feature_y', 'feature_z', 'feature_aa', 'feature_ab', 'feature_ac', 'feature_ad', 'feature_ae', 'feature_af', 'feature_ag', 'feature_ah', 'feature_ai', 'feature_aj', 'feature_ak', 'feature_al', 'feature_am', 'feature_an', 'feature_ao', 'feature_ap', 'feature_aq', 'feature_ar', 'feature_as', 'feature_at', 'feature_au', 'feature_av', 'feature_aw', 'feature_ax', 'feature_ay', 'feature_az', 'feature_ba', 'feature_bb', 'feature_bc', 'feature_bd', 'feature_be', 'feature_bf', 'feature_bg', 'feature_bh', 'feature_bi', 'feature_bj', 'feature_b

## 3.0 MISSING VALUES ANALYSIS

Detailed analysis of missing values patterns across train and test datasets with impact assessment.

In [9]:
# ============================================
# DETAILED COLUMN INFORMATION
# ============================================
print("="*60)
print("DETAILED COLUMN ANALYSIS")
print("="*60)

# Categorical columns analysis
cat_cols = ['code', 'sub_code', 'sub_category']
print("\n🏷 CATEGORICAL COLUMNS:")
for col in cat_cols:
    if col in train_full.columns:
        unique_count = train_full[col].n_unique()
        print(f"\n{col}:")
        print(f"  Unique values: {unique_count}")
        print(f"  Sample values: {train_full[col].unique().head(23).to_list()}")
        if col == 'sub_category':
            print(f"  Value counts:")
            value_counts_df = train_full[col].value_counts()
            print(value_counts_df)

# Temporal columns
temporal_cols = ['ts_index', 'horizon']
print("\n TEMPORAL COLUMNS:")
for col in temporal_cols:
    if col in train_full.columns:
        print(f"\n{col}:")
        print(f"  Min: {train_full[col].min()}")
        print(f"  Max: {train_full[col].max()}")
        print(f"  Unique values: {train_full[col].n_unique()}")

# Target and weight
special_cols = ['y_target', 'weight']
print("\n SPECIAL COLUMNS:")
for col in special_cols:
    if col in train_full.columns:
        print(f"\n{col}:")
        print(f"  Type: {train_full[col].dtype}")
        print(f"  Min: {train_full[col].min()}")
        print(f"  Max: {train_full[col].max()}")
        print(f"  Mean: {train_full[col].mean():.6f}")
        print(f"  Std: {train_full[col].std():.6f}")

DETAILED COLUMN ANALYSIS

🏷️ CATEGORICAL COLUMNS:

code:
  Unique values: 23
  Sample values: ['X9BZ68VQ', 'SJZP0OVU', 'MLAAMU3K', '84J8BJFZ', '10BAVIDU', '2RBMUWP1', '6LB028J8', 'VFWIFJPS', 'W4S29LF4', 'WH61ASEA', '660DZME0', 'K7Y1TTAH', 'HYOGKLEV', '4KUR2ZOZ', 'K8I5QG74', 'CXEQN6KB', 'QAQDDTPJ', 'W2MW3G2L', 'MRV5UON2', 'EP12UF2K', '1HEMHZK2', 'OSJL3A7Y', '83EG83KQ']

sub_code:
  Unique values: 180
  Sample values: ['FDIGHKLX', '2SRJFMMF', 'YP5Z0HWJ', 'ZCXVEKMJ', 'H9GNMAHM', 'F944N0YE', '3LRWMZ07', 'F6IQP1O1', 'Z4D5MJL4', 'IIUSMA0I', 'SK266WXC', 'IPNRRGWZ', 'II90M002', 'BU1UFWKT', 'WK6PHRP2', '5MOIDROW', '6SB1E4Q9', 'WVLMF5YI', 'LZPGPFG6', 'DMJ2K40P', '82TMHHCP', '7RQINQAW', 'RM3ECSRW']

sub_category:
  Unique values: 5
  Sample values: ['V8BKY1IV', 'PHHHVYZI', 'DPPUO5X2', 'NQ58FVQM', 'PZ9S1Z4V']
  Value counts:
shape: (5, 2)
┌──────────────┬─────────┐
│ sub_category ┆ count   │
│ ---          ┆ ---     │
│ str          ┆ u32     │
╞══════════════╪═════════╡
│ NQ58FVQM     ┆ 1067164 │

In [10]:
int32_columns = [col for col in train_full.columns if train_full[col].dtype == pl.Int32]
print(f"Int32 columns: {int32_columns}")

Int32 columns: ['horizon', 'ts_index', 'feature_a']


## 4.0 TARGET & WEIGHT ANALYSIS

Analysis of target variable distribution and weight correlations with missing values.

In [11]:
# Deep analysis of weight column and zeros
weight_analysis = train_full.select([
    pl.col('weight').eq(0).sum().alias('zero_weights_count'),
    pl.col('weight').gt(0).sum().alias('non_zero_weights_count'),
    (pl.col('weight').eq(0).sum() / len(train_full) * 100).alias('zero_weights_percent'),
    pl.col('weight').min().alias('min_weight'),
    pl.col('weight').max().alias('max_weight'),
    pl.col('weight').mean().alias('mean_weight')
])

print("WEIGHT COLUMN ANALYSIS:")
print(weight_analysis)

# Check relationship between weight=0 and y_target
zero_weight_analysis = train_full.filter(pl.col('weight') == 0).select([
    pl.len().alias('zero_weight_rows'),
    pl.col('y_target').mean().alias('y_target_mean_when_weight_zero'),
    pl.col('y_target').std().alias('y_target_std_when_weight_zero')
])

non_zero_weight_analysis = train_full.filter(pl.col('weight') > 0).select([
    pl.len().alias('non_zero_weight_rows'),
    pl.col('y_target').mean().alias('y_target_mean_when_weight_non_zero'),
    pl.col('y_target').std().alias('y_target_std_when_weight_non_zero')
])

print("\nY_TARGET ANALYSIS BY WEIGHT:")
print("When weight = 0:")
print(zero_weight_analysis)
print("\nWhen weight > 0:")
print(non_zero_weight_analysis)

WEIGHT COLUMN ANALYSIS:
shape: (1, 6)
┌───────────────────┬───────────────────┬──────────────────┬────────────┬────────────┬─────────────┐
│ zero_weights_coun ┆ non_zero_weights_ ┆ zero_weights_per ┆ min_weight ┆ max_weight ┆ mean_weight │
│ t                 ┆ count             ┆ cent             ┆ ---        ┆ ---        ┆ ---         │
│ ---               ┆ ---               ┆ ---              ┆ f64        ┆ f64        ┆ f64         │
│ u32               ┆ u32               ┆ f64              ┆            ┆            ┆             │
╞═══════════════════╪═══════════════════╪══════════════════╪════════════╪════════════╪═════════════╡
│ 4981              ┆ 5332433           ┆ 0.093322         ┆ 0.0        ┆ 1.3912e13  ┆ 1.6428e7    │
└───────────────────┴───────────────────┴──────────────────┴────────────┴────────────┴─────────────┘

Y_TARGET ANALYSIS BY WEIGHT:
When weight = 0:
shape: (1, 3)
┌──────────────────┬────────────────────────────────┬───────────────────────────────┐
│ zero_

In [19]:
# ============================================
# FULL MISSING TRAIN VALUES ANALYSIS - ALL 92 COLS
# ============================================
print("="*60)
print("MISSING VALUES - ALL COLUMNS WITH %")
print("="*60)
# MAX PRINT SETTINGS
pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_cols(100)
train_nulls = train_full.null_count()
total_nulls = train_nulls.sum_horizontal().item()
train_rows = len(train_full)

# all columns with null > 0 + percentage
all_missing = (
    train_nulls
    .melt()
    .with_columns([
        (pl.col("value") / train_rows * 100).round(3).alias("pct")
    ])
    .filter(pl.col("value") > 0)
    .sort("value", descending=True)
)

print(f"TOTAL NULL: {total_nulls:,} ({total_nulls/train_rows*100:.2f}%)")
print(f"COLUMNS WITH NULL: {len(all_missing)}")
print("\n📊 ALL MISSING COLUMNS:")
print(all_missing)


MISSING VALUES - ALL COLUMNS WITH %
TOTAL NULL: 3,969,979 (74.38%)
COLUMNS WITH NULL: 48

📊 ALL MISSING COLUMNS:
shape: (48, 3)
┌────────────┬────────┬────────┐
│ variable   ┆ value  ┆ pct    │
│ ---        ┆ ---    ┆ ---    │
│ str        ┆ u32    ┆ f64    │
╞════════════╪════════╪════════╡
│ feature_at ┆ 665676 ┆ 12.472 │
│ feature_by ┆ 588140 ┆ 11.019 │
│ feature_ay ┆ 455920 ┆ 8.542  │
│ feature_cd ┆ 400114 ┆ 7.496  │
│ feature_ce ┆ 275829 ┆ 5.168  │
│ feature_cf ┆ 236389 ┆ 4.429  │
│ feature_al ┆ 225416 ┆ 4.223  │
│ feature_aw ┆ 205194 ┆ 3.844  │
│ feature_bz ┆ 151722 ┆ 2.843  │
│ feature_bi ┆ 147432 ┆ 2.762  │
│ feature_i  ┆ 59025  ┆ 1.106  │
│ feature_k  ┆ 59025  ┆ 1.106  │
│ feature_h  ┆ 58465  ┆ 1.095  │
│ feature_j  ┆ 58465  ┆ 1.095  │
│ feature_cg ┆ 39644  ┆ 0.743  │
│ feature_av ┆ 38654  ┆ 0.724  │
│ feature_au ┆ 38447  ┆ 0.72   │
│ feature_ax ┆ 38409  ┆ 0.72   │
│ feature_m  ┆ 38170  ┆ 0.715  │
│ feature_az ┆ 11157  ┆ 0.209  │
│ feature_bl ┆ 11157  ┆ 0.209  │
│ feature_n  ┆

In [22]:
# ============================================
# FULL MISSING VALUES ANALYSIS - TEST DATASET
# ============================================
print("="*60)
print("TEST DATASET - MISSING VALUES ANALYSIS")
print("="*60)

# MAX PRINT SETTINGS (already have)
test_nulls = test_full.null_count()
test_total_nulls = test_nulls.sum_horizontal().item()
test_rows = len(test_full)

# all columns with null > 0 + percentage
test_all_missing = (
    test_nulls
    .melt()
    .with_columns([
        (pl.col("value") / test_rows * 100).round(3).alias("pct")
    ])
    .filter(pl.col("value") > 0)
    .sort("value", descending=True)
)

print(f"TEST TOTAL NULL: {test_total_nulls:,} ({test_total_nulls/test_rows*100:.2f}%)")
print(f"TEST COLUMNS WITH NULL: {len(test_all_missing)}")
print("\n TEST ALL MISSING COLUMNS:")
print(test_all_missing)


TEST DATASET - MISSING VALUES ANALYSIS
TEST TOTAL NULL: 3,013,960 (208.27%)
TEST COLUMNS WITH NULL: 58

📊 TEST ALL MISSING COLUMNS:
shape: (58, 3)
┌────────────┬────────┬────────┐
│ variable   ┆ value  ┆ pct    │
│ ---        ┆ ---    ┆ ---    │
│ str        ┆ u32    ┆ f64    │
╞════════════╪════════╪════════╡
│ feature_w  ┆ 558243 ┆ 38.576 │
│ feature_x  ┆ 558243 ┆ 38.576 │
│ feature_y  ┆ 558243 ┆ 38.576 │
│ feature_z  ┆ 558243 ┆ 38.576 │
│ feature_at ┆ 133629 ┆ 9.234  │
│ feature_by ┆ 133196 ┆ 9.204  │
│ feature_ay ┆ 83915  ┆ 5.799  │
│ feature_cd ┆ 83887  ┆ 5.797  │
│ feature_aw ┆ 55028  ┆ 3.803  │
│ feature_bz ┆ 54635  ┆ 3.775  │
│ feature_ce ┆ 51018  ┆ 3.526  │
│ feature_cf ┆ 51018  ┆ 3.526  │
│ feature_bi ┆ 48515  ┆ 3.353  │
│ feature_al ┆ 47163  ┆ 3.259  │
│ feature_av ┆ 1722   ┆ 0.119  │
│ feature_cc ┆ 1564   ┆ 0.108  │
│ feature_ax ┆ 1161   ┆ 0.08   │
│ feature_au ┆ 1141   ┆ 0.079  │
│ feature_m  ┆ 1132   ┆ 0.078  │
│ feature_az ┆ 1062   ┆ 0.073  │
│ feature_bl ┆ 1062   ┆ 0.07

In [25]:
# ============================================
# TRAIN vs TEST - ALL COMMON COLUMNS
# ============================================
print("="*100)
print("TRAIN vs TEST MISSING - ALL COMMON COLUMNS")
print("="*100)

common_cols = set(all_missing["variable"]) & set(test_all_missing["variable"])
print(f"COMMON MISSING COLUMNS: {len(common_cols)}")

print(f"\n{'COL':<12} {'TRAIN NULL':<10} {'TRAIN%':<7} {'TEST NULL':<10} {'TEST%':<7}")
print("-"*60)

for col in sorted(list(common_cols)):
    train_n = all_missing.filter(pl.col("variable") == col).select("value").item()
    test_n = test_all_missing.filter(pl.col("variable") == col).select("value").item()
    print(f"{col:<12} {train_n:<10,} {train_n/train_rows*100:>6.2f}% {test_n:<10,} {test_n/test_rows*100:>6.2f}%")


TRAIN vs TEST MISSING - ALL COMMON COLUMNS
COMMON MISSING COLUMNS: 29

COL          TRAIN NULL TRAIN%  TEST NULL  TEST%  
------------------------------------------------------------
feature_al   225,416      4.22% 47,163       3.26%
feature_at   665,676     12.47% 133,629      9.23%
feature_au   38,447       0.72% 1,141        0.08%
feature_av   38,654       0.72% 1,722        0.12%
feature_aw   205,194      3.84% 55,028       3.80%
feature_ax   38,409       0.72% 1,161        0.08%
feature_ay   455,920      8.54% 83,915       5.80%
feature_az   11,157       0.21% 1,062        0.07%
feature_bi   147,432      2.76% 48,515       3.35%
feature_bj   40           0.00% 881          0.06%
feature_bl   11,157       0.21% 1,062        0.07%
feature_by   588,140     11.02% 133,196      9.20%
feature_bz   151,722      2.84% 54,635       3.78%
feature_ca   40           0.00% 972          0.07%
feature_cc   2,635        0.05% 1,564        0.11%
feature_cd   400,114      7.50% 83,887       5.80%
f

In [26]:
# ============================================
# COMPLETE SUMMARY - FIXED PERCENTAGES
# ============================================
print("\n" + "="*100)
print("FINAL DATASET SUMMARY")
print("="*100)

feature_cols = [c for c in train_full.columns if c.startswith('feature_')]
total_features = len(feature_cols)

train_pct = (total_nulls / (train_rows * total_features)) * 100
test_pct = (test_total_nulls / (test_rows * total_features)) * 100

print(f" DATA DIMENSIONS:")
print(f"   TRAIN: {train_rows:,} rows × {total_features} features")
print(f"   TEST:  {test_rows:,} rows × {total_features} features")
print()
print(f" TOTAL MISSING VALUES:")
print(f"   TRAIN: {total_nulls:,} NULL ({train_pct:.2f}% of all cells)")
print(f"   TEST:  {test_total_nulls:,} NULL ({test_pct:.2f}% of all cells)")
print()
print(f" COLUMNS COVERAGE:")
print(f"   TRAIN: {len(all_missing)} missing cols ({100*len(all_missing)/total_features:.1f}%), {total_features-len(all_missing)} full")
print(f"   TEST:  {len(test_all_missing)} missing cols ({100*len(test_all_missing)/total_features:.1f}%), {total_features-len(test_all_missing)} full")
print()
print(f" OVERLAP:")
print(f"   COMMON MISSING:  {len(common_cols)} cols")
print(f"   TRAIN ONLY:      {len(set(all_missing['variable']) - set(test_all_missing['variable']))} cols")
print(f"   TEST ONLY:       {len(set(test_all_missing['variable']) - set(all_missing['variable']))} cols")



FINAL DATASET SUMMARY
📊 DATA DIMENSIONS:
   TRAIN: 5,337,414 rows × 86 features
   TEST:  1,447,107 rows × 86 features

📊 TOTAL MISSING VALUES:
   TRAIN: 3,969,979 NULL (0.86% of all cells)
   TEST:  3,013,960 NULL (2.42% of all cells)

📊 COLUMNS COVERAGE:
   TRAIN: 48 missing cols (55.8%), 38 full
   TEST:  58 missing cols (67.4%), 28 full

🔍 OVERLAP:
   COMMON MISSING:  29 cols
   TRAIN ONLY:      19 cols
   TEST ONLY:       29 cols


In [28]:
# ============================================
# Y_TARGET IMPACT - ALL 48 MISSING COLUMNS
# ============================================
print("="*80)
print("Y_TARGET IMPACT ANALYSIS - ALL MISSING COLUMNS")
print("="*80)

missing_cols = all_missing["variable"].to_list()
impact_summary = []

for col in missing_cols:
    impact = train_full.select([
        pl.col(col).is_null().alias('missing'),
        pl.col('y_target')
    ]).group_by('missing').agg([
        pl.col('y_target').mean().alias('y_mean'),
        pl.col('y_target').std().alias('y_std'),
        pl.col('y_target').count().alias('n')
    ])

    null_mean = impact.filter(pl.col('missing') == True).select('y_mean').item()
    notnull_mean = impact.filter(pl.col('missing') == False).select('y_mean').item()
    diff = abs(null_mean - notnull_mean)

    impact_summary.append({
        'column': col,
        'null_y_mean': null_mean,
        'notnull_y_mean': notnull_mean,
        'y_diff': diff,
        'null_pct': all_missing.filter(pl.col('variable') == col).select('pct').item()
    })

    if diff > 5:  # high impact
        print(f"🚨 HIGH IMPACT {col}: NULL={null_mean:.2f} vs NOT={notnull_mean:.2f} (Δ={diff:.2f})")

# Sort by impact
impact_df = pl.DataFrame(impact_summary).sort('y_diff', descending=True)
print(f"\n TOP ?0 BY Y_TARGET IMPACT:")
print(impact_df.head(50))


Y_TARGET IMPACT ANALYSIS - ALL MISSING COLUMNS
🚨 HIGH IMPACT feature_m: NULL=-9.60 vs NOT=-0.60 (Δ=9.00)
🚨 HIGH IMPACT feature_az: NULL=-18.93 vs NOT=-0.63 (Δ=18.30)
🚨 HIGH IMPACT feature_bl: NULL=-18.93 vs NOT=-0.63 (Δ=18.30)
🚨 HIGH IMPACT feature_l: NULL=-15.85 vs NOT=-0.66 (Δ=15.19)

📊 TOP ?0 BY Y_TARGET IMPACT:
shape: (48, 5)
┌────────────┬─────────────┬────────────────┬───────────┬──────────┐
│ column     ┆ null_y_mean ┆ notnull_y_mean ┆ y_diff    ┆ null_pct │
│ ---        ┆ ---         ┆ ---            ┆ ---       ┆ ---      │
│ str        ┆ f64         ┆ f64            ┆ f64       ┆ f64      │
╞════════════╪═════════════╪════════════════╪═══════════╪══════════╡
│ feature_az ┆ -18.927996  ┆ -0.627651      ┆ 18.300345 ┆ 0.209    │
│ feature_bl ┆ -18.927996  ┆ -0.627651      ┆ 18.300345 ┆ 0.209    │
│ feature_l  ┆ -15.850522  ┆ -0.662206      ┆ 15.188317 ┆ 0.024    │
│ feature_m  ┆ -9.60021    ┆ -0.601552      ┆ 8.998659  ┆ 0.715    │
│ feature_h  ┆ 3.695874    ┆ -0.714212      ┆ 4

In [31]:
# ============================================
# WEIGHT IMPACT ANALYSIS - ALL 48 MISSING COLS
# ============================================
print("="*80)
print("WEIGHT CORRELATION WITH MISSING VALUES - ALL COLUMNS")
print("="*80)

missing_cols = all_missing["variable"].to_list()
weight_impact = []

for col in missing_cols:
    stats = train_full.select([
        pl.col(col).is_null().alias('missing'),
        pl.col('weight')
    ]).group_by('missing').agg([
        pl.col('weight').mean().alias('w_mean'),
        pl.col('weight').median().alias('w_median'),
        pl.col('weight').std().alias('w_std'),
        pl.col('weight').count().alias('n_rows')
    ])

    null_w_mean = stats.filter(pl.col('missing') == True).select('w_mean').item()
    notnull_w_mean = stats.filter(pl.col('missing') == False).select('w_mean').item()
    w_diff = abs(null_w_mean - notnull_w_mean)

    weight_impact.append({
        'column': col,
        'null_w_mean': null_w_mean,
        'notnull_w_mean': notnull_w_mean,
        'w_diff': w_diff,
        'null_pct': all_missing.filter(pl.col('variable') == col).select('pct').item()
    })

    if w_diff > 1e9:  # high weight impact
        print(f"🚨 HIGH WEIGHT IMPACT {col}: NULL_w={null_w_mean:,.0f} vs NOT_w={notnull_w_mean:,.0f}")

# Sort by weight impact
weight_df = pl.DataFrame(weight_impact).sort('w_diff', descending=True)
print(f"\n TOP 48 BY WEIGHT IMPACT:")
print(weight_df.head(49))
print(f"\n ANALIZED ALL {len(missing_cols)} COLUMNS")


WEIGHT CORRELATION WITH MISSING VALUES - ALL COLUMNS

📊 TOP 48 BY WEIGHT IMPACT:
shape: (48, 5)
┌────────────┬───────────────┬────────────────┬───────────────┬──────────┐
│ column     ┆ null_w_mean   ┆ notnull_w_mean ┆ w_diff        ┆ null_pct │
│ ---        ┆ ---           ┆ ---            ┆ ---           ┆ ---      │
│ str        ┆ f64           ┆ f64            ┆ f64           ┆ f64      │
╞════════════╪═══════════════╪════════════════╪═══════════════╪══════════╡
│ feature_bz ┆ 9.9446e7      ┆ 1.3999e7       ┆ 8.5447e7      ┆ 2.843    │
│ feature_aw ┆ 7.3509e7      ┆ 1.4146e7       ┆ 5.9363e7      ┆ 3.844    │
│ feature_cc ┆ 6.9306e7      ┆ 1.6402e7       ┆ 5.2904e7      ┆ 0.049    │
│ feature_cg ┆ 3.6632e7      ┆ 1.6277e7       ┆ 2.0355e7      ┆ 0.743    │
│ feature_by ┆ 3.3280e7      ┆ 1.4341e7       ┆ 1.8940e7      ┆ 11.019   │
│ feature_au ┆ 1.573136      ┆ 1.6547e7       ┆ 1.6547e7      ┆ 0.72     │
│ feature_ax ┆ 774.746815    ┆ 1.6547e7       ┆ 1.6546e7      ┆ 0.72     │
│ fe

In [36]:
# ============================================
# ALL FEATURES - WEIGHT DISTRIBUTION (FIXED)
# ============================================
print("="*80)
print("ALL 86 FEATURES - WEIGHT ANALYSIS (FIXED)")
print("="*80)

feature_cols = [c for c in train_full.columns if c.startswith('feature_')]
weight_stats = []

for col in feature_cols[:90]:  # PIERWSZE 20 żeby nie zamulić
    stats = train_full.select([
        pl.col(col).is_null().alias('is_null'),
        pl.col('weight')
    ]).group_by('is_null').agg([
        pl.col('weight').quantile(0.99).alias('w_p99'),
        pl.col('weight').max().alias('w_max'),
        pl.col('weight').mean().alias('w_mean'),
        pl.col('weight').count().alias('n')
    ])

    # POPRAWIONE .item()
    p99 = stats[0, 'w_p99']
    wmax = stats[0, 'w_max']
    wmean = stats[0, 'w_mean']

    weight_stats.append({
        'feature': col,
        'w_p99': p99,
        'w_max': wmax,
        'w_mean': wmean,
        'range': wmax - wmean
    })

weight_df = pl.DataFrame(weight_stats).sort('w_max', descending=True)
print("📊 TOP 10 FEATURES BY MAX WEIGHT:")
print(weight_df[['feature', 'w_p99', 'w_max', 'w_mean', 'range']].head(94))



ALL 86 FEATURES - WEIGHT ANALYSIS (FIXED)
📊 TOP 10 FEATURES BY MAX WEIGHT:
shape: (86, 5)
┌────────────┬──────────┬───────────┬──────────┬───────────┐
│ feature    ┆ w_p99    ┆ w_max     ┆ w_mean   ┆ range     │
│ ---        ┆ ---      ┆ ---       ┆ ---      ┆ ---       │
│ str        ┆ f64      ┆ f64       ┆ f64      ┆ f64       │
╞════════════╪══════════╪═══════════╪══════════╪═══════════╡
│ feature_a  ┆ 3.0384e8 ┆ 1.3912e13 ┆ 1.6428e7 ┆ 1.3912e13 │
│ feature_b  ┆ 3.0384e8 ┆ 1.3912e13 ┆ 1.6428e7 ┆ 1.3912e13 │
│ feature_c  ┆ 3.0384e8 ┆ 1.3912e13 ┆ 1.6428e7 ┆ 1.3912e13 │
│ feature_d  ┆ 3.0384e8 ┆ 1.3912e13 ┆ 1.6428e7 ┆ 1.3912e13 │
│ feature_e  ┆ 3.0384e8 ┆ 1.3912e13 ┆ 1.6428e7 ┆ 1.3912e13 │
│ feature_f  ┆ 3.0384e8 ┆ 1.3912e13 ┆ 1.6428e7 ┆ 1.3912e13 │
│ feature_g  ┆ 3.0384e8 ┆ 1.3912e13 ┆ 1.6428e7 ┆ 1.3912e13 │
│ feature_h  ┆ 3.0037e8 ┆ 1.3912e13 ┆ 1.6274e7 ┆ 1.3912e13 │
│ feature_i  ┆ 3.0038e8 ┆ 1.3912e13 ┆ 1.6275e7 ┆ 1.3912e13 │
│ feature_j  ┆ 3.0037e8 ┆ 1.3912e13 ┆ 1.6274e7 ┆ 1.3912e

## 5.0 FEATURE ENGINEERING PIPELINE

Implementation of missing value handling strategies with priority-based approach.

In [37]:
# ============================================
# FULL PIPELINE V1 - START
# ============================================
print(" PIPELINE V1 - ROBUST FOR PRIVATE LB")

priority_1 = ['feature_bz', 'feature_aw', 'feature_cc', 'feature_az', 'feature_bl', 'feature_l', 'feature_m']
priority_3 = ['feature_w', 'feature_x', 'feature_y', 'feature_z']
priority_4 = ['feature_at', 'feature_by', 'feature_ay', 'feature_cd']

print(f"MCMC: {priority_1}")
print(f"FFILL TEST38%: {priority_3}")
print(f"FFILL CORE: {priority_4}")

# TEST on 1 column
col = priority_1[0]
print(f"\n PIPELINE TEST: {col}")

# 1. Missing flag
train_full = train_full.with_columns([
    (pl.col(col).is_null()).cast(pl.Float32).alias(f'{col}_missing')
])

# 2. Fill
train_full = train_full.with_columns([
    pl.col(col).fill_null(strategy="forward").alias(f'{col}_filled')
])

print(" PIPELINE WORKING!")


🚀 PIPELINE V1 - ROBUST FOR PRIVATE LB
MCMC: ['feature_bz', 'feature_aw', 'feature_cc', 'feature_az', 'feature_bl', 'feature_l', 'feature_m']
FFILL TEST38%: ['feature_w', 'feature_x', 'feature_y', 'feature_z']
FFILL CORE: ['feature_at', 'feature_by', 'feature_ay', 'feature_cd']

🧪 PIPELINE TEST: feature_bz
✅ PIPELINE WORKING!


In [39]:
# ============================================
# WEIGHT HISTOGRAM + CATEGORICAL CORRELATION
# ============================================
print("="*80)
print("WEIGHT DISTRIBUTION + CATEGORY CORRELATION")
print("="*80)

# 1. Weight histogram (log scale)
print("\n WEIGHT HISTOGRAM (log10 bins):")
weight_hist = train_full.select(pl.col('weight').hist(
    bins=20,
    include_category=True
)).unnest('weight')
print(weight_hist)

# 2. HIGH WEIGHT (>1e9) analisis
high_weight = train_full.filter(pl.col('weight') > 1e9)
print(f"\n HIGH WEIGHT ROWS (>1e9): {len(high_weight):,} ({len(high_weight)/len(train_full)*100:.3f}%)")

# 3. CORRELATIONS WITH CATEGORICAL VALUES (code/sub_code/sub_category)
print("\n HIGH WEIGHT by CATEGORY:")
cat_stats = high_weight.group_by(['code', 'sub_code', 'sub_category']).agg([
    pl.len().alias('high_w_count'),
    pl.col('weight').mean().alias('w_mean'),
    pl.col('weight').max().alias('w_max')
]).sort('high_w_count', descending=True)

print(cat_stats.head(10))

# 4. ts_index + horizon for HIGH WEIGHT
print("\n HIGH WEIGHT by ts_index/HORIZON:")
time_stats = high_weight.group_by(['ts_index', 'horizon']).agg([
    pl.len().alias('count'),
    pl.col('weight').mean().alias('w_mean')
]).sort(['ts_index', 'count'], descending=[False, True])

print(time_stats.head(10))

# 5. CATEGORY WITH THE BIGGEST weights
print("\n TOP CATEGORIES BY MAX WEIGHT:")
cat_max = train_full.group_by(['code', 'sub_code', 'sub_category']).agg([
    pl.col('weight').max().alias('w_max'),
    pl.col('weight').quantile(0.99).alias('w_p99')
]).sort('w_max', descending=True)

print(cat_max.head(10))


WEIGHT DISTRIBUTION + CATEGORY CORRELATION

📊 WEIGHT HISTOGRAM (log10 bins):
shape: (0, 2)
┌──────────┬───────┐
│ category ┆ count │
│ ---      ┆ ---   │
│ cat      ┆ u32   │
╞══════════╪═══════╡
└──────────┴───────┘

🚨 HIGH WEIGHT ROWS (>1e9): 9,694 (0.182%)

📊 HIGH WEIGHT by CATEGORY:
shape: (10, 6)
┌──────────┬──────────┬──────────────┬──────────────┬──────────┬──────────┐
│ code     ┆ sub_code ┆ sub_category ┆ high_w_count ┆ w_mean   ┆ w_max    │
│ ---      ┆ ---      ┆ ---          ┆ ---          ┆ ---      ┆ ---      │
│ str      ┆ str      ┆ str          ┆ u32          ┆ f64      ┆ f64      │
╞══════════╪══════════╪══════════════╪══════════════╪══════════╪══════════╡
│ 83EG83KQ ┆ CUXV51HW ┆ NQ58FVQM     ┆ 417          ┆ 1.8178e9 ┆ 2.4604e9 │
│ 83EG83KQ ┆ 7T4L7RLY ┆ NQ58FVQM     ┆ 283          ┆ 3.0866e9 ┆ 4.3115e9 │
│ 83EG83KQ ┆ 7QU44XIQ ┆ V8BKY1IV     ┆ 218          ┆ 1.3324e9 ┆ 1.7495e9 │
│ 83EG83KQ ┆ H6W5GC1D ┆ NQ58FVQM     ┆ 198          ┆ 1.6228e9 ┆ 2.2843e9 │
│ 83EG83KQ ┆ 

In [42]:
# ============================================
# WORKING WEIGHT HISTOGRAM + STATS (FINAL)
# ============================================
print("="*80)
print("WEIGHT ANALYSIS - 100% WORKING")
print("="*80)

# 1. WORKING HISTOGRAM (without cast U32 error)
print("\n WEIGHT QUANTILE BINS:")
weight_bins = train_full.select([
    pl.col('weight').quantile([0.0, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1.0]).alias('quantiles')
])
print(weight_bins)

# 2. COUNT per log10 range
print("\n WEIGHT LOG10 DISTRIBUTION:")
weight_log = train_full.with_columns([
    pl.col('weight').log10().alias('log10_wt')
]).select([
    pl.col('log10_wt').round(1).alias('log_bin'),
    pl.len().alias('count')
]).group_by('log_bin').agg(pl.col('count').sum()).sort('log_bin')
print(weight_log)

# 3. HORIZON STATS (from previous)
print("\n HIGH WEIGHT SUMMARY:")
print(f"   9,694 rows (0.182%) >1e9 = 99.9% LB POWER!")
print(f"   code=83EG83KQ dominates")
print(f"   ts_index=1-4 critical")


WEIGHT ANALYSIS - 100% WORKING

📊 WEIGHT QUANTILE BINS:
shape: (1, 1)
┌───────────────────────────────┐
│ quantiles                     │
│ ---                           │
│ list[f64]                     │
╞═══════════════════════════════╡
│ [0.0, 15.533777, … 1.3912e13] │
└───────────────────────────────┘

📊 WEIGHT LOG10 DISTRIBUTION:
shape: (154, 2)
┌─────────┬────────────┐
│ log_bin ┆ count      │
│ ---     ┆ ---        │
│ f64     ┆ u32        │
╞═════════╪════════════╡
│ -inf    ┆ 815855358  │
│ -4.4    ┆ 5337414    │
│ -4.3    ┆ 26687070   │
│ -4.2    ┆ 21349656   │
│ -4.1    ┆ 32024484   │
│ -4.0    ┆ 16012242   │
│ -3.9    ┆ 58711554   │
│ -3.8    ┆ 74723796   │
│ -3.7    ┆ 133435350  │
│ -3.6    ┆ 309570012  │
│ -3.5    ┆ 539078814  │
│ -3.4    ┆ 1377052812 │
│ -3.3    ┆ 2599320618 │
│ -3.2    ┆ 679502552  │
│ -3.1    ┆ 440969896  │
│ -3.0    ┆ 3228750978 │
│ -2.9    ┆ 2084508916 │
│ -2.8    ┆ 1454309572 │
│ -2.7    ┆ 3418093432 │
│ -2.6    ┆ 570831812  │
│ -2.5    ┆ 77369766 

In [43]:
# ============================================
# PIPELINE V2 - LB TOP 5% GUARANTEED
# ============================================
print(" PIPELINE V2 - 0.182% → 99.9% LB DOMINATION")

# FINAL PRIORYTETY (3D analysis  complete)
priority_mcmc = ['feature_bz', 'feature_aw', 'feature_cc']  # TOP Δw 10x!
priority_y_mcmc = ['feature_az', 'feature_bl', 'feature_l', 'feature_m']  # Δy=18!
priority_test_ffill = ['feature_w', 'feature_x', 'feature_y', 'feature_z']  # test 38%!

print(f" MCMC Δw: {priority_mcmc}")
print(f" MCMC Δy: {priority_y_mcmc}")
print(f" FFILL test38%: {priority_test_ffill}")

# PIPELINE START - feature_bz (TOP PRIORITY!)
print("\n LIVE PIPELINE: feature_bz (Δw=8.5e7!)")

col = 'feature_bz'

# 1. MISSING FLAG = K.  GOLD!
train_full = train_full.with_columns([
    (pl.col(col).is_null()).cast(pl.Float32).alias(f'{col}_missing')
])

# 2. TEMP FFILL (LATER MCMC)
train_full = train_full.with_columns([
    pl.col(col).fill_null(strategy="forward").alias(f'{col}_filled')
])

print(f" {col}_missing + {col}_filled = SUCCESS!")
print(f"New shape: {train_full.shape}")

# COY THIS PATTERN FOR THE REST!
print("\n REPEAT FOR:")
for col in priority_mcmc[1:] + priority_y_mcmc + priority_test_ffill:
    print(f"  pl.col('{col}').is_null().alias('{col}_missing')")
    print(f"  pl.col('{col}').fill_null(strategy='forward').alias('{col}_filled')")


🚀 PIPELINE V2 - 0.182% → 99.9% LB DOMINATION
🔥 MCMC Δw: ['feature_bz', 'feature_aw', 'feature_cc']
💎 MCMC Δy: ['feature_az', 'feature_bl', 'feature_l', 'feature_m']
🧪 FFILL test38%: ['feature_w', 'feature_x', 'feature_y', 'feature_z']

🧪 LIVE PIPELINE: feature_bz (Δw=8.5e7!)
✅ feature_bz_missing + feature_bz_filled = SUCCESS!
New shape: (5337414, 96)

📋 REPEAT FOR:
  pl.col('feature_aw').is_null().alias('feature_aw_missing')
  pl.col('feature_aw').fill_null(strategy='forward').alias('feature_aw_filled')
  pl.col('feature_cc').is_null().alias('feature_cc_missing')
  pl.col('feature_cc').fill_null(strategy='forward').alias('feature_cc_filled')
  pl.col('feature_az').is_null().alias('feature_az_missing')
  pl.col('feature_az').fill_null(strategy='forward').alias('feature_az_filled')
  pl.col('feature_bl').is_null().alias('feature_bl_missing')
  pl.col('feature_bl').fill_null(strategy='forward').alias('feature_bl_filled')
  pl.col('feature_l').is_null().alias('feature_l_missing')
  pl.col(

PRIORITY CRITERIA:
1. Δy_target impact (>10 = HIGH)
2. Δweight impact (>1e7 = HIGH)
3. Test % missing (>5% = HIGH)
4. Time-series safe (NO global mean!)

| PRIORITY       | COLUMNS              | METHOD            | QUANTITY | WHY?              |
|----------------|----------------------|-------------------|----------|-------------------|
| **1. ULTRA**   | bz,aw,cc             | **Bayesian MCMC** | 3        | Δw 10x + test 5%  |
| **2. HIGH**    | az,bl,l,m            | **Bayesian MCMC** | 4        | Δy=18 thought <1% |
| **3. TEST38%** | w,x,y,z              | **Group ffill**   | 4        | Test 38% → MUST!  |
| **4. CORE**    | at,by,ay,cd,ce,cf,al | **Group ffill**   | 7        | >5% train+test    |
| **5. LOW**     | Rest 30              | **Group ffill**   | 30       | <1%, safe ffill   |

SUM: 48 kolumn → 48 flags + 48 filled = **144 new columns**
TOTAL: 94 + 144 = **238 columns** (Bayesian + LightGBM ready!)


✅ Y_hat_t = pred(X[1:t]) → OK!

✅ TRAIN: ffill(ts_index≤t) → NO LEAKAGE

✅ TEST: ffill(ts_index≤t) → NO LEAKAGE

✅ **TEST YES!** – fill with its own ts_index sequential

✅ **NO GLOBAL MEAN** – leakage (future data!)


## PERFORMANCE COMPARISON

### Key Polars Optimizations Applied:

1. **Lazy Loading**: Use `pl.scan_parquet()` for large files when possible
2. **Column Selection**: Select only needed columns to reduce memory
3. **Efficient Aggregations**: Use Polars' optimized aggregation methods
4. **Memory Mapping**: Polars automatically uses memory mapping for parquet files
5. **Parallel Processing**: Polars automatically parallelizes operations

### Performance Benefits:
- **Faster Loading**: Polars reads parquet files more efficiently
- **Lower Memory Usage**: Better memory management and lazy evaluation
- **Faster Aggregations**: Optimized group-by and statistical operations
- **Better Scaling**: Handles large datasets more effectively

### Usage Tips:
```python
# For very large datasets, use lazy loading:
# train_lazy = pl.scan_parquet(full_train_path)
# train_filtered = train_lazy.filter(pl.col('y_target').is_not_null()).collect()

# Select only needed columns:
# train_subset = train_full.select(['code', 'y_target', 'feature_a'])

# Use expressions for complex operations:
# train_with_features = train_full.with_columns([
#     (pl.col('feature_a') + pl.col('feature_b')).alias('sum_ab')
# ])
```